<a href="https://colab.research.google.com/github/jabri62018/Jabri_lab/blob/Jabri_lab/Zx_Planck.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:

# file= Zx_Planck.ipynb
# Author= Eng. Abdulla Al-Jabri
# Zero input: Planck time to Hubble tension

import mpmath as mp
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import zipfile
from IPython.display import display, HTML

print("="*60)
print("PROBLEM: Connect Planck time to Hubble tension using Zx zeros")
print("="*60)

mp.mp.dps = 80
xp = 21.0

def Zx(t):
    x = mp.mpc('0.5', str(t))
    return mp.exp(-x/xp) * mp.exp(-5*mp.log(x)) * mp.log(x) * mp.sin(2*mp.pi/x)

def find_zeros(n=30):
    mp.mp.dps = 60
    t_vals = np.arange(14.0, 200, 0.02)
    f_vals = [float(mp.im(Zx(t))) for t in t_vals]
    brackets = []
    for j in range(len(f_vals)-1):
        if f_vals[j] * f_vals[j+1] < 0:
            brackets.append((t_vals[j], t_vals[j+1]))
    mp.mp.dps = 80
    zeros = []
    for t_min, t_max in brackets[:n]:
        r = mp.findroot(lambda tt: mp.im(Zx(tt)), (t_min, t_max), tol=mp.mpf('1e-65'))
        zeros.append(float(r))
    return zeros

zeros = find_zeros(30)

CONSTANTS = {
    't_P': 5.391e-44, 'l_P': 1.616e-35, 'h': 6.626e-34,
    'G': 6.674e-11, 'c': 299792458,
    't_H': 4.35e17, 'H0': 67.4, 'DE': 6.9e-27,
}

def calc_C(gamma):
    h = mp.mpf('1e-15')
    t = mp.mpf(gamma)
    z = Zx(t)
    zppp = (Zx(t+2*h) - 2*Zx(t+h) + 2*Zx(t-h) - Zx(t-2*h)) / (2*h)
    return float(0.5 * gamma**2 * mp.re(zppp / z))

rows = []
for i, g in enumerate(zeros, 1):
    C = calc_C(g)
    logC = np.log10(abs(C) + 1e-300)
    diffs = {k: abs(logC - np.log10(abs(v) + 1e-300)) for k,v in CONSTANTS.items()}
    matched = min(diffs, key=diffs.get)
    rows.append({'Root': i, 'gamma': g, 'C_calc': C, 'Matched': matched,
                 'Value': CONSTANTS[matched], 'Log Diff': diffs[matched]})

df = pd.DataFrame(rows)
df.to_csv('Zx_Planck_match.csv', index=False, float_format='%.15e')

plt.figure(figsize=(12,6))
plt.loglog(df['C_calc'], df['gamma'], 'o-', markersize=6)
for _, r in df.iterrows():
    if r['Matched'] in ['t_P', 't_H', 'H0', 'h', 'DE']:
        plt.annotate(r['Matched'], (r['C_calc'], r['gamma']), fontsize=10, weight='bold')
plt.axvline(CONSTANTS['t_P'], color='red', linestyle='--', label='Planck time')
plt.axvline(CONSTANTS['t_H'], color='blue', linestyle='--', label='Hubble time')
plt.xlabel('C_calc')
plt.ylabel('gamma zero')
plt.title('Zx: Planck Time to Hubble Tension')
plt.legend()
plt.grid(True, which='both', alpha=0.3)
plt.savefig('Zx_Planck_plot.png', dpi=300)
plt.close()

print("\n=== TABLE: First 15 roots ===")
display(df[['Root','gamma','C_calc','Matched','Log Diff']].head(15))
display(HTML('<img src="Zx_Planck_plot.png" width="900">'))

with zipfile.ZipFile('Zx_Planck_results.zip', 'w') as zipf:
    zipf.write('Zx_Planck_match.csv')
    zipf.write('Zx_Planck_plot.png')
from google.colab import files
files.download('Zx_Planck_results.zip')
print("\nDone. Files downloaded.")

PROBLEM: Connect Planck time to Hubble tension using Zx zeros

=== TABLE: First 15 roots ===


,Root,gamma,C_calc,Matched,Log Diff
0,1,15.053925,-9.367783e-30,DE,2.867212
1,2,74.140130,-4.959475e-31,h,2.874184
2,3,138.859904,1.708707e-30,h,3.411416


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


Done. Files downloaded.
